In [ ]:
"""
Consider the dataset house_pricing.csv
a. Design following model for house prices data set to predict the house price and use k-fold validation
   (Grid Search) to test the following model
   a. XGBoostRegressor(learning rate = [0.001, 0.3], max_depth=[2,5], n_estimators=[10,20])
   b. DecisionTreeRegressor(max_depth=[3,None], min_sample_split=[2,5,10], min_sample_leaf=[1,5] )
   c. RandomForestRegressor(max_features=[3,4,5])
b. Mention the best r2_score in each case and also mention which model is the best among the above tried models.

"""

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

# Load dataset
df = pd.read_csv(r"/home/sarthakredasani/Documents/CDAC_ML/Datasets/Datasets/Housing.csv")
# Assume "Price" is the target variable
X = df.drop(columns=["price"])
y = df["price"]

# Identify categorical columns
categorical_cols = ["driveway", "recroom", "fullbase", "gashw", "airco", "prefarea"]

# Convert categorical columns to category dtype for XGBoost compatibility
for col in categorical_cols:
    X[col] = X[col].astype("category")

# One-Hot Encode categorical columns for DecisionTree and RandomForest
encoder = OneHotEncoder(drop="first", sparse_output=False)
X_encoded = encoder.fit_transform(X[categorical_cols])
X_encoded_df = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out())

# Concatenate encoded features with numeric features
X = pd.concat([X.drop(columns=categorical_cols), X_encoded_df], axis=1)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# K-Fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Hyperparameter grids
xgb_params = {"learning_rate": [0.001, 0.3], "max_depth": [2,5], "n_estimators": [10,20]}
dt_params = {"max_depth": [3, None], "min_samples_split": [2,5,10], "min_samples_leaf": [1,5]}
rf_params = {"max_features": [3,4,5]}

# Define models with GridSearchCV
xgb = GridSearchCV(XGBRegressor(enable_categorical=True), xgb_params, cv=kf, scoring="r2", n_jobs=-1)
dt = GridSearchCV(DecisionTreeRegressor(), dt_params, cv=kf, scoring="r2", n_jobs=-1)
rf = GridSearchCV(RandomForestRegressor(), rf_params, cv=kf, scoring="r2", n_jobs=-1)

# Fit models
xgb.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Get best estimators and predictions
xgb_best = xgb.best_estimator_
dt_best = dt.best_estimator_
rf_best = rf.best_estimator_

xgb_r2 = r2_score(y_test, xgb_best.predict(X_test))
dt_r2 = r2_score(y_test, dt_best.predict(X_test))
rf_r2 = r2_score(y_test, rf_best.predict(X_test))

# Display results
print(f"XGBoost Best R2 Score: {xgb_r2:.4f}")
print(f"Decision Tree Best R2 Score: {dt_r2:.4f}")
print(f"Random Forest Best R2 Score: {rf_r2:.4f}")

best_model = max([(xgb_r2, "XGBoost"), (dt_r2, "Decision Tree"), (rf_r2, "Random Forest")], key=lambda x: x[0])
print(f"Best Model: {best_model[1]} with R2 Score {best_model[0]:.4f}")


XGBoost Best R2 Score: 0.6232
Decision Tree Best R2 Score: 0.4993
Random Forest Best R2 Score: 0.5638
Best Model: XGBoost with R2 Score 0.6232


In [11]:
"""   
consider the dataset glass.csv
a. Build a K-nearest neighbor classifier with hyper parameter values n_neighbors=3 and metrics = 'manhattan' with Response=Type. 
   find accuracy of the model with K-Fold CV
b. Find the outliers in the dataset of the features(exclude Type) using Isolation Forest Algorithm with contamination = 0.5
c. Build a support vector classifier with linear kernel, with one versus rest of all classification and c=50 
    for glass.csv and find accuracy score with K-Fold CV. Compare it with k-nearest neighbor
d. Build a Random Forest Classifier with max_features=3 with K-Fold CV. Compare it with K-nearest neighbor and also tht of SVM.

"""

"   \nconsider the dataset glass.csv\na. Build a K-nearest neighbor classifier with hyper parameter values n_neighbors=3 and metrics = 'manhattan' with Response=Type. \n   find accuracy of the model with K-Fold CV\nb. Find the outliers in the dataset of the features(exclude Type) using Isolation Forest Algorithm with contamination = 0.5\nc. Build a support vector classifier with linear kernel, with one versus rest of all classification and c=50 \n    for glass.csv and find accuracy score with K-Fold CV. Compare it with k-nearest neighbor\nd. Build a Random Forest Classifier with max_features=3 with K-Fold CV. Compare it with K-nearest neighbor and also tht of SVM.\n\n"

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.metrics import r2_score
from sklearn.compose import *    
from sklearn.svm import *
from sklearn.ensemble import *
from sklearn.preprocessing import *
from sklearn.neighbors import *


df = pd.read_csv(r"/home/sarthakredasani/Documents/CDAC_ML/Cases/Cases/Glass Identification/Glass.csv")
df.head(5)
X, y = df.drop('Type', axis =1), df['Type']
scaler = StandardScaler()
X_scaler = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

knn = KNeighborsClassifier(n_neighbors=3, metric='manhattan' )
knn_accuracy = np.mean(cross_val_score(knn, X_scaler, y, cv= kf, scoring="accuracy"))
print(f"KNN Accuracy: {knn_accuracy:.4f}")


# Outlier Detection using Isolation Forest

iso_forest = IsolationForest(contamination=0.5, random_state=42)
outliers = iso_forest.fit_predict(X_scaler)

# Count outliers (-1 denotes outliers)
num_outliers = sum(outliers == -1)
print(f"Number of Outliers Detected: {num_outliers}")




# Define SVM with linear kernel and One-vs-Rest strategy
svm = SVC(kernel="linear", C=50)

# Cross-validation accuracy
svm_accuracy = np.mean(cross_val_score(svm, X_scaler, y, cv=kf, scoring="accuracy"))
print(f"SVM Accuracy: {svm_accuracy:.4f}")

# Compare with KNN
print(f"SVM vs KNN: {'SVM performs better' if svm_accuracy > knn_accuracy else 'KNN performs better'}")



# Define Random Forest with max_features=3
rf = RandomForestClassifier(max_features=3, random_state=42)

# Cross-validation accuracy
rf_accuracy = np.mean(cross_val_score(rf, X_scaler, y, cv=kf, scoring="accuracy"))
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")

# Compare models
best_model = max([(knn_accuracy, "KNN"), (svm_accuracy, "SVM"), (rf_accuracy, "Random Forest")], key=lambda x: x[0])
print(f"Best Model: {best_model[1]} with Accuracy {best_model[0]:.4f}")




KNN Accuracy: 0.7053
Number of Outliers Detected: 107
SVM Accuracy: 0.6495
SVM vs KNN: KNN performs better
Random Forest Accuracy: 0.7803
Best Model: Random Forest with Accuracy 0.7803
